# Hello, Deep Learning — a neural network on handwritten digits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/production-ai-engineering/blob/main/notebooks/hello_deeplearning.ipynb)

Pairs with **ML_Study_14**. Trains a real **neural network** (`MLPClassifier` — a multi-layer perceptron)
on the digits dataset. Runs **anywhere, offline** — no GPU, no download. The PyTorch/MNIST version is at
the bottom for Colab.

## 1. The data — 8×8 handwritten digits (0–9)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
%matplotlib inline

digits = load_digits()
print("images:", digits.images.shape, " labels:", digits.target.shape)   # 1797 of 8x8
fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. The network — a multi-layer perceptron (784→… but here 64→64→32→10)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = digits.data / 16.0            # flatten 8x8 -> 64 features, scale to 0..1
y = digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

# hidden_layer_sizes = the hidden layers; relu = activation; adam = gradient descent
net = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu",
                    solver="adam", max_iter=400, random_state=0)
net.fit(Xtr, ytr)                 # the whole train loop (forward/loss/backprop/update) runs here
print("layers:", [64] + list(net.hidden_layer_sizes) + [10])

## 3. Evaluate — accuracy on the held-out test set

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
pred = net.predict(Xte)
print(f"test accuracy: {accuracy_score(yte, pred):.1%}")
print("\nconfusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(yte, pred))

## 4. Watch it learn — the loss curve

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(net.loss_curve_, color="#1f77b4")
plt.title("Loss going DOWN each epoch = the network learning (gradient descent)")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.tight_layout(); plt.show()

## 5. Predict a few digits

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 6, figsize=(10, 2.2))
idx = np.arange(6)
for ax, k in zip(axes, idx):
    ax.imshow(Xte[k].reshape(8,8), cmap="gray_r")
    ax.set_title(f"pred {net.predict([Xte[k]])[0]}\ntrue {yte[k]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. The same idea in PyTorch on real MNIST (run this in Colab)

Everything above is a neural net; here is the framework version on full 28×28 MNIST. Every line maps to
a concept in ML_Study_14 — PyTorch is just the training loop, automated.

```python
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

train = DataLoader(datasets.MNIST(".", train=True,  download=True, transform=transforms.ToTensor()),
                   batch_size=64, shuffle=True)
test  = DataLoader(datasets.MNIST(".", train=False, download=True, transform=transforms.ToTensor()),
                   batch_size=1000)

model = nn.Sequential(nn.Flatten(), nn.Linear(784,128), nn.ReLU(), nn.Linear(128,10))
loss_fn, opt = nn.CrossEntropyLoss(), torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    for images, labels in train:
        opt.zero_grad()
        loss = loss_fn(model(images), labels)   # forward + loss
        loss.backward(); opt.step()             # backprop + update

correct = total = 0
with torch.no_grad():
    for images, labels in test:
        correct += (model(images).argmax(1) == labels).sum().item(); total += labels.size(0)
print(f"MNIST test accuracy: {100*correct/total:.1f}%")   # ~97%
```

## Takeaway
- A neural network is layers of neurons trained by the loop: **forward → loss → backprop → update**.
- `MLPClassifier` (scikit-learn) and PyTorch do the *same thing* — PyTorch just scales to big data + GPUs.
- On tiny 8×8 digits an MLP already nails it; the ideas scale straight up to MNIST and beyond.